# CS490 Capstone Project: Vehicle MPG Analysis

**Dataset:** [Vehicle MPG (1984-2023)](https://www.kaggle.com/datasets/mexwell/vehicle-mpg-1984-2023)

## Run Instructions
1. Download the dataset from Kaggle.
2. Rename the file to `VehicleMPG.csv` and place it in this notebook folder.
3. Run this notebook top-to-bottom.
4. Generated plots are saved to the `Figures/` folder.

## Research Question
What vehicle characteristics most influence fuel efficiency in modern vehicles, and how has fuel efficiency improved over time?

In [82]:
# CS490 Capstone Project: Vehicle MPG Analysis
# ------------------------------------------------
# This notebook follows a full data science workflow:
# 1) load and inspect data,
# 2) clean/filter to match project scope,
# 3) perform exploratory analysis,
# 4) train/evaluate ML models,
# 5) answer the research question with evidence.
#
# Dataset source:
# https://www.kaggle.com/datasets/mexwell/vehicle-mpg-1984-2023
#
# GitHub repository:
# https://github.com/bendery13/IntensivePythonCapstone
#
# Before running:
# - Place VehicleMPG.csv in the same folder as this notebook
# - Run all cells from top to bottom for reproducible results
# - Create a "Figures" subfolder to store generated plots (or adjust save_figure function accordingly)

In [83]:
# Section 1: Setup and data loading for a fully reproducible run.
# This cell centralizes all imports and global settings so the notebook is easier to rerun

import warnings
from pathlib import Path

# Suppress non-critical warnings so output stays readable for grading/review
warnings.filterwarnings("ignore")

# Core numerical/data libraries
import numpy as np
import pandas as pd

# Visualization libraries
import seaborn as sns
import matplotlib.pyplot as plt

# Scikit-learn utilities for preprocessing, modeling, and evaluation
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make plots visually consistent across all cells
sns.set_theme(style="whitegrid")

# Show all columns when printing dataframe summaries
pd.set_option("display.max_columns", None)

# Create an output folder for figures so notebook files stay lightweight
FIGURES_DIR = Path("Figures")
FIGURES_DIR.mkdir(exist_ok=True)


def save_figure(filename, fig=None, dpi=300):
    # Save a matplotlib figure to the Figures folder with consistent export settings
    # fig=None means "use the current active figure"
    if fig is None:
        fig = plt.gcf()
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    print(f"Saved figure: {output_path}")


# Load source data
csv_path = "VehicleMPG.csv"
df = pd.read_csv(csv_path)

# Check on dataset size before deeper analysis
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

# Preview first few rows to verify columns loaded correctly
df.head()

Rows: 45,896 | Columns: 26


,ID,Model Year,Make,Model,Estimated Annual Petrolum Consumption (Barrels),Fuel Type 1,City MPG (Fuel Type 1),Highway MPG (Fuel Type 1),Combined MPG (Fuel Type 1),Fuel Type 2,City MPG (Fuel Type 2),Highway MPG (Fuel Type 2),Combined MPG (Fuel Type 2),Engine Cylinders,Engine Displacement,Drive,Engine Description,Transmission,Vehicle Class,Time to Charge EV (hours at 120v),Time to Charge EV (hours at 240v),Range (for EV),City Range (for EV - Fuel Type 1),City Range (for EV - Fuel Type 2),Hwy Range (for EV - Fuel Type 1),Hwy Range (for EV - Fuel Type 2)
0,1,1985,Alfa Romeo,Spider Veloce 2000,14.167143,Regular Gasoline,19,25,21,NaN,0,0,0,4.0,2.0,Rear-Wheel Drive,(FFS),Manual 5-spd,Two Seaters,0,0.0,0,0.0,0.0,0.0,0.0
1,2,1985,Bertone,X1/9,13.523182,Regular Gasoline,20,26,22,NaN,0,0,0,4.0,1.5,Rear-Wheel Drive,NaN,Manual 5-spd,Two Seaters,0,0.0,0,0.0,0.0,0.0,0.0
2,3,1985,Chevrolet,Corvette,17.500588,Regular Gasoline,15,21,17,NaN,0,0,0,8.0,5.7,Rear-Wheel Drive,(350 V8) (FFS),Automatic 4-spd,Two Seaters,0,0.0,0,0.0,0.0,0.0,0.0
3,4,1985,Chevrolet,Corvette,17.500588,Regular Gasoline,15,20,17,NaN,0,0,0,8.0,5.7,Rear-Wheel Drive,(350 V8) (FFS),Manual 4-spd,Two Seaters,0,0.0,0,0.0,0.0,0.0,0.0
4,5,1985,Nissan,300ZX,18.594375,Regular Gasoline,15,18,16,NaN,0,0,0,6.0,3.0,Rear-Wheel Drive,"(GUZZLER) (FFS,TRBO)",Automatic 4-spd,Two Seaters,0,0.0,0,0.0,0.0,0.0,0.0


## 1) Data Inspection and Quality Checks

This section verifies structure, data types, missingness, and duplicate rows that could affect analysis quality.

In [84]:
# Data inspection: structure, missing values, and duplicates
# This gives context for data quality decisions made later

# Show column names, data types, non-null counts, and memory usage
df.info()

# Count missing values per column and sort from highest to lowest
missing = df.isna().sum().sort_values(ascending=False)
print("\nTop columns by missing values:")

# Display only columns that actually have missing data
display(missing[missing > 0].head(15))

# Check duplicate rows across all columns
duplicate_rows = df.duplicated().sum()
print(f"Duplicate rows in raw dataset: {duplicate_rows}")

<class 'pandas.DataFrame'>
RangeIndex: 45896 entries, 0 to 45895
Data columns (total 26 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   ID                                               45896 non-null  int64  
 1   Model Year                                       45896 non-null  int64  
 2   Make                                             45896 non-null  str    
 3   Model                                            45896 non-null  str    
 4   Estimated Annual Petrolum Consumption (Barrels)  45896 non-null  float64
 5   Fuel Type 1                                      45896 non-null  str    
 6   City MPG (Fuel Type 1)                           45896 non-null  int64  
 7   Highway MPG (Fuel Type 1)                        45896 non-null  int64  
 8   Combined MPG (Fuel Type 1)                       45896 non-null  int64  
 9   Fuel Type 2                            

Fuel Type 2            44059
Engine Description     17031
Drive                   1186
Engine Cylinders         487
Engine Displacement      485
Transmission              11
dtype: int64

Duplicate rows in raw dataset: 0


## 2) Scope Filtering and Feature Preparation

To align with the proposal, I focus on modern non-electric vehicles (2000+) and select variables relevant to MPG analysis and modeling.

In [85]:
# Main fuel descriptor column used for EV exclusion logic
fuel_col = "Fuel Type 1"

# Define non-EV condition using two checks:
# 1) Fuel description does not mention electric
# 2) EV range is zero (additional guard against edge cases)
non_ev_mask = (
    ~df[fuel_col].fillna("").str.contains("electric", case=False)
    & (df["Range (for EV)"].fillna(0) == 0)
)

# Keep only model year 2000+ and non-EV rows
modern_df = df[(df["Model Year"] >= 2000) & non_ev_mask].copy()

# Select features used in EDA and machine learning
selected_cols = [
    "Model Year", "Make", "Model", "Fuel Type 1",
    "City MPG (Fuel Type 1)", "Highway MPG (Fuel Type 1)", "Combined MPG (Fuel Type 1)",
    "Engine Cylinders", "Engine Displacement", "Transmission", "Drive", "Vehicle Class"
]

# Keep selected columns and remove perfect duplicate rows
modern_df = modern_df[selected_cols].drop_duplicates().copy()

# Re-check missingness after filtering, because profile can change by subgroup
missing_after_filter = modern_df.isna().sum().sort_values(ascending=False)
print("Top missing columns after filtering:")
display(missing_after_filter[missing_after_filter > 0].head(10))

# Ensure target column is non-missing before modeling
rows_before_target_drop = len(modern_df)
modern_df = modern_df.dropna(subset=["Combined MPG (Fuel Type 1)"]).copy()
rows_removed = rows_before_target_drop - len(modern_df)
print(f"Rows removed for missing target MPG: {rows_removed}")

# Report final analysis dataset size
print(f"Filtered dataset shape: {modern_df.shape}")

# Preview filtered dataset
modern_df.head()

Top missing columns after filtering:


Series([], dtype: int64)

Rows removed for missing target MPG: 0
Filtered dataset shape: (26942, 12)


,Model Year,Make,Model,Fuel Type 1,City MPG (Fuel Type 1),Highway MPG (Fuel Type 1),Combined MPG (Fuel Type 1),Engine Cylinders,Engine Displacement,Transmission,Drive,Vehicle Class
15587,2000,Acura,NSX,Premium Gasoline,15,22,18,6.0,3.0,Automatic 4-spd,Rear-Wheel Drive,Two Seaters
15588,2000,Acura,NSX,Premium Gasoline,15,22,18,6.0,3.2,Manual 6-spd,Rear-Wheel Drive,Two Seaters
15589,2000,BMW,M Coupe,Premium Gasoline,17,23,19,6.0,3.2,Manual 5-spd,Rear-Wheel Drive,Two Seaters
15590,2000,BMW,Z3 Coupe,Premium Gasoline,17,24,19,6.0,2.8,Automatic 4-spd,Rear-Wheel Drive,Two Seaters
15591,2000,BMW,Z3 Coupe,Premium Gasoline,17,24,19,6.0,2.8,Manual 5-spd,Rear-Wheel Drive,Two Seaters


## 2b) NumPy Numerical Summary

Before renaming columns, we use NumPy directly on the raw array to compute basic statistics. This demonstrates NumPy's role as the numerical foundation underlying pandas and scikit-learn.

In [86]:
# Convert pandas Series to a NumPy array for direct numerical operations
combined = modern_df["Combined MPG (Fuel Type 1)"].to_numpy()

# Central tendency and spread for combined MPG
print(f"Mean combined MPG: {np.mean(combined):.2f}")
print(f"Median combined MPG: {np.median(combined):.2f}")
print(f"STD combined MPG: {np.std(combined):.2f}")

Mean combined MPG: 21.34
Median combined MPG: 21.00
STD combined MPG: 5.65


## 3) Exploratory Data Analysis (EDA)

These visuals summarize MPG distributions and relationships between MPG and key engine variables.

In [87]:
# Rename long source column names for cleaner downstream analysis code
# This makes plotting/modeling code easier to read and less error-prone
modern_df = modern_df.rename(columns={
    "Fuel Type 1": "fuel_type",
    "City MPG (Fuel Type 1)": "city_mpg",
    "Highway MPG (Fuel Type 1)": "highway_mpg",
    "Combined MPG (Fuel Type 1)": "combined_mpg",
    "Engine Cylinders": "cylinders",
    "Engine Displacement": "displacement",
    "Model Year": "model_year",
    "Vehicle Class": "vehicle_class",
    "Transmission": "transmission",
    "Drive": "drive"
})

# Mixed-type descriptive summary (numeric + categorical)
# Useful for checking value ranges, cardinality, and potential anomalies
modern_df.describe(include="all").T.head(12)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
model_year,26942.0,NaN,NaN,NaN,2011.98237,6.74634,2000.0,2006.0,2012.0,2018.0,2023.0
Make,26942,75,Chevrolet,2056,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Model,26942,3406,Jetta,144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fuel_type,26942,5,Regular Gasoline,14580,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city_mpg,26942.0,NaN,NaN,NaN,18.905761,5.582707,7.0,15.0,18.0,21.0,58.0
highway_mpg,26942.0,NaN,NaN,NaN,25.54094,5.96907,11.0,21.0,25.0,29.0,61.0
combined_mpg,26942.0,NaN,NaN,NaN,21.34307,5.650552,8.0,18.0,21.0,24.0,59.0
cylinders,26942.0,NaN,NaN,NaN,5.802836,1.829229,2.0,4.0,6.0,6.0,16.0
displacement,26942.0,NaN,NaN,NaN,3.299031,1.31491,0.6,2.2,3.0,4.0,8.4
transmission,26942,34,Automatic 4-spd,3918,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [88]:
# EDA figures: three separate plots saved individually for clarity

# Figure 1: Distribution of combined MPG to understand center/spread/skew
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(modern_df["combined_mpg"], kde=True, ax=ax)
ax.set_title("Combined MPG Distribution")
ax.set_xlabel("Combined MPG")
ax.set_ylabel("Vehicle Count")
plt.tight_layout()
save_figure("eda_mpg_distribution.png", fig)
plt.close(fig)

# Figure 2: Continuous relationship between engine size and fuel efficiency
# alpha lowers point opacity to reduce overplotting on dense regions
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=modern_df, x="displacement", y="combined_mpg", alpha=0.3, ax=ax)
ax.set_title("Displacement vs Combined MPG")
ax.set_xlabel("Engine Displacement (Liters)")
ax.set_ylabel("Combined MPG")
plt.tight_layout()
save_figure("eda_displacement_vs_mpg.png", fig)
plt.close(fig)

# Figure 3: MPG distribution by cylinder category (median + variability)
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=modern_df, x="cylinders", y="combined_mpg", ax=ax)
ax.set_title("Cylinders vs Combined MPG")
ax.set_xlabel("Engine Cylinders")
ax.set_ylabel("Combined MPG")
plt.tight_layout()
save_figure("eda_cylinders_vs_mpg.png", fig)
plt.close(fig)

Saved figure: Figures\eda_mpg_distribution.png
Saved figure: Figures\eda_displacement_vs_mpg.png
Saved figure: Figures\eda_cylinders_vs_mpg.png


### EDA Interpretation

- Combined MPG is not uniformly distributed, with most vehicles concentrated in a middle-efficiency band.
- Larger engine displacement generally corresponds to lower combined MPG.
- Vehicles with more cylinders tend to have lower median MPG, supporting expected engine-efficiency tradeoffs.

In [89]:
# Correlation analysis among primary numeric variables
# Correlation helps quantify linear direction/strength between feature pairs
numeric_cols = ["model_year", "city_mpg", "highway_mpg", "combined_mpg", "cylinders", "displacement"]

# Compute Pearson correlation matrix on selected numeric columns
corr = modern_df[numeric_cols].corr(numeric_only=True)

# Render heatmap with annotations for precise coefficient reading
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True, ax=ax)
ax.set_title("Correlation Matrix")
ax.set_xlabel("Features")
ax.set_ylabel("Features")

# Save and close
save_figure("eda_correlation_matrix.png", fig)
plt.close(fig)

Saved figure: Figures\eda_correlation_matrix.png


### Correlation Interpretation

The heatmap confirms the expected relationships: engine displacement (−0.74) and cylinder count (−0.70) have strong negative correlations with combined MPG — larger, more powerful engines are less fuel-efficient. Model year shows a modest positive correlation (+0.31), consistent with the trend that newer vehicles tend to be slightly more efficient. 

## 4) Time Trend Analysis

This section tests whether modern vehicle MPG has improved over model years.

In [90]:
# Time trend analysis: average combined MPG by model year
# This directly tests the "has MPG improved over time" portion of the question
mpg_by_year = (
    modern_df.groupby("model_year", as_index=False)["combined_mpg"]
    .mean()
    .rename(columns={"combined_mpg": "avg_combined_mpg"})
)

# Line plot is appropriate for ordered temporal trends
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=mpg_by_year, x="model_year", y="avg_combined_mpg", marker="o", ax=ax)
ax.set_title("Average Combined MPG by Model Year (2000+ Non-EV)")
ax.set_xlabel("Model Year")
ax.set_ylabel("Average Combined MPG")

# Export trend figure for report usage
save_figure("time_mpg_by_year.png", fig)
plt.close(fig)

# Show most recent years as a quick textual check of recent movement
mpg_by_year.tail()

Saved figure: Figures\time_mpg_by_year.png


,model_year,avg_combined_mpg
19,2019,23.489969
20,2020,23.424771
21,2021,23.174877
22,2022,23.291806
23,2023,22.969605


### Time Trend Interpretation

Average combined MPG rises from around 20 MPG in 2000 to a peak near 23–24 MPG in the late 2010s before leveling off slightly in recent years. This confirms that modern non-EV fuel efficiency has broadly improved over the study period, though gains appear to have slowed after ~2015.

## 5) Machine Learning Models

Two supervised regression models are trained and compared:
- Linear Regression (interpretable baseline)
- Random Forest Regressor (nonlinear baseline)

In [91]:
# Machine learning section: train and compare two supervised regression models
# The target is numeric (combined MPG), so regression is the correct problem type
model_df = modern_df.copy()

# Predict combined MPG without city/highway MPG to reduce direct leakage
# city_mpg and highway_mpg are too close to combined_mpg by definition
features = ["model_year", "cylinders", "displacement", "fuel_type", "transmission", "drive", "vehicle_class"]
target = "combined_mpg"

# Build feature matrix X and target vector y
X = model_df[features]
y = model_df[target]

# Separate numeric and categorical features for appropriate preprocessing
numeric_features = ["model_year", "cylinders", "displacement"]
categorical_features = ["fuel_type", "transmission", "drive", "vehicle_class"]

# Numeric preprocessing: median imputation for any missing numeric values
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

# Categorical preprocessing:
# - fill missing with most frequent category
# - one-hot encode to convert categories into model-ready numeric columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ColumnTransformer applies each preprocessing pipeline to the right column subset
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# Hold-out split for unbiased evaluation on unseen data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline linear model (interpretable, fast, commonly used benchmark)
linreg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Nonlinear ensemble model to capture complex relationships/interactions
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

# Model registry for loop-based training/evaluation
models = {
    "Linear Regression": linreg_model,
    "Random Forest": rf_model,
}

# Collect metrics and predictions for each model
metrics_rows = []
predictions = {}

for name, model in models.items():
    # Fit model on training data only
    model.fit(X_train, y_train)

    # Predict on test data for fair performance reporting
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    # Compute core regression metrics:
    # MAE: average absolute error
    # RMSE: penalizes larger errors more strongly
    # R2: proportion of variance explained
    metrics_rows.append({
        "model": name,
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": mean_squared_error(y_test, y_pred) ** 0.5,
        "r2": r2_score(y_test, y_pred),
    })

# Sort by RMSE (lower is better) to rank model performance
metrics_df = pd.DataFrame(metrics_rows).sort_values(by="rmse")
display(metrics_df)

# Store top model name for downstream diagnostic plots
best_model_name = metrics_df.iloc[0]["model"]
print(f"Best model by RMSE: {best_model_name}")

,model,mae,rmse,r2
1,Random Forest,0.833252,1.484747,0.935411
0,Linear Regression,1.723058,2.628496,0.797572


Best model by RMSE: Random Forest


In [92]:
# Diagnostic evaluation for the best-performing model
# This provides visual error analysis beyond summary metrics

# Retrieve test predictions from whichever model ranked best by RMSE
best_pred = predictions[best_model_name]

# Build a compact results table for plotting and inspection.
results_df = pd.DataFrame({"actual": y_test.values, "predicted": best_pred})
results_df["residual"] = results_df["actual"] - results_df["predicted"]

# Figure 1: Actual vs Predicted scatter.
# If points cluster near the diagonal, predictions are close to true values
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=results_df, x="actual", y="predicted", alpha=0.35, ax=ax)
lims = [results_df[["actual", "predicted"]].min().min(), results_df[["actual", "predicted"]].max().max()]
ax.plot(lims, lims, "r--")
ax.set_title(f"Actual vs Predicted MPG ({best_model_name})")
ax.set_xlabel("Actual Combined MPG")
ax.set_ylabel("Predicted Combined MPG")
plt.tight_layout()
save_figure("model_actual_vs_predicted.png", fig)
plt.close(fig)

# Figure 2: Residual distribution.
# A well-behaved model usually has residuals centered around zero
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(results_df["residual"], kde=True, ax=ax)
ax.set_title("Residual Distribution")
ax.set_xlabel("Actual - Predicted MPG")
ax.set_ylabel("Count")
plt.tight_layout()
save_figure("model_residuals.png", fig)
plt.close(fig)

# Show first few rows for a quick numerical sanity check
results_df.head()

Saved figure: Figures\model_actual_vs_predicted.png
Saved figure: Figures\model_residuals.png


,actual,predicted,residual
0,26,22.734596,3.265404
1,21,20.380000,0.620000
2,26,27.582518,-1.582518
3,12,14.433708,-2.433708
4,21,22.785292,-1.785292


### Model Evaluation Interpretation

Use the scatter plot to check how closely predictions follow the ideal diagonal line. Use the residual histogram to verify that errors are centered near zero and to spot bias or large outliers.

## 6) Final Conclusions

This analysis focused on non-electric vehicles from model year 2000 onward and addressed the question: **Which characteristics most influence MPG, and how has MPG changed over time?**

- **Most influential characteristics:** Engine displacement and cylinder count show clear negative relationships with MPG in both the scatter/box plots and the correlation heatmap. Vehicle class, transmission, drive type, and fuel type also improve predictive performance in the machine learning models.
- **Change over time:** Average combined MPG generally increases over the 2000+ period, indicating broad long-term improvement in fuel efficiency, with mild year-to-year variation in recent years.
- **Modeling results:** The Random Forest model outperformed Linear Regression on the held-out test set.
  - Random Forest: MAE = 0.833, RMSE = 1.485, R² = 0.935
  - Linear Regression: MAE = 1.723, RMSE = 2.628, R² = 0.798
- **Direct answer to research question:** Fuel efficiency in modern vehicles is strongly tied to engine design variables (especially displacement and cylinder count), and overall efficiency has improved over time in the studied period.